# Notebook 07b: RL Agent Training — DQN (PyTorch) and PPO (Stable-Baselines3)

**MSc Dissertation: Agentic AI for Sovereign Risk Assessment under Climate-Related Fiscal Stress**

This notebook trains two independent reinforcement learning agents on all 9 profiles of the
Sovereign Risk Gymnasium environment:

1. **DQN (Deep Q-Network)** — implemented from scratch in PyTorch to demonstrate low-level
   understanding of Q-learning, experience replay, and target networks (Mnih et al., 2015).
2. **PPO (Proximal Policy Optimisation)** — implemented via Stable-Baselines3, a production-grade
   library providing a well-tested PPO implementation (Schulman et al., 2017).

The dual-algorithm approach enables **cross-algorithmic consistency validation**: findings that
appear in both DQN and PPO are more robust than those from a single algorithm.

### Why DQN and PPO?
- **DQN** is a value-based method: it learns Q(s,a) — the expected return from each (state, action)
  pair — and acts greedily with respect to this estimate. It is well-suited to discrete action spaces.
- **PPO** is a policy-gradient method: it directly optimises a stochastic policy π(a|s) using the
  advantage function. The clipping mechanism (PPO's key innovation) prevents the large policy updates
  that destabilise earlier policy-gradient methods (TRPO).

Both are trained for **100,000 timesteps per profile** — sufficient for convergence analysis on this
low-dimensional environment, though partial convergence is noted where it occurs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pickle
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
warnings.filterwarnings('ignore')

from src.environment.sovereign_risk_env import SovereignRiskEnv
from src.environment.config import list_profiles
from src.agents.dqn import DQNAgent, train_dqn

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback

# Paths
CONFIG_PATH  = '../data/processed/transition_parameters.json'
SCALING_PATH = '../data/processed/scaling_parameters.json'
MODELS_DIR   = '../outputs/models'
RESULTS_DIR  = '../outputs/results'
FIGURES_DIR  = '../outputs/figures'
for d in [MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

PROFILES          = list_profiles(CONFIG_PATH)
TOTAL_TIMESTEPS   = 100_000
DEVICE            = 'cuda' if torch.cuda.is_available() else 'cpu'

def make_env(profile, seed=SEED):
    return SovereignRiskEnv(
        profile=profile, config_path=CONFIG_PATH,
        scaling_path=SCALING_PATH, seed=seed
    )

print(f'torch version : {torch.__version__}')
print(f'device        : {DEVICE}')
print(f'profiles      : {len(PROFILES)}')
print(f'timesteps/profile: {TOTAL_TIMESTEPS:,}')
print(f'total training steps: {TOTAL_TIMESTEPS * len(PROFILES) * 2:,} (DQN + PPO combined)')

---
## DQN: Algorithm Explanation

### The Q-Learning Framework

The core idea: for each state $s$ and action $a$, maintain an estimate $Q(s,a)$ of the expected
discounted future reward. The optimal policy is then simply $\pi^*(s) = \arg\max_a Q(s,a)$.

The **Bellman equation** gives us the self-consistency condition that the true $Q^*$ must satisfy:
$$Q^*(s,a) = r + \gamma \max_{a'} Q^*(s', a')$$

where $r$ is the immediate reward, $\gamma$ is the discount factor, and $s'$ is the next state.
This becomes the training target.

### Key DQN Innovations (Mnih et al., 2015)

1. **Function approximation**: Replace the Q-table with a neural network $Q(s,a;\theta)$.
2. **Experience replay**: Store $(s,a,r,s',d)$ tuples; train on random mini-batches to break
   temporal correlations.
3. **Target network**: A slowly-updated copy $Q(s,a;\theta^-)$ provides stable regression targets,
   preventing the "chasing your own tail" instability of naive Q-learning with neural nets.
4. **Epsilon-greedy exploration**: $\varepsilon$ decays from 1 (random) to 0.05 (mostly greedy)
   over training.

### Applied to Fiscal Policy

In our setting:
- $s$ = 7-dim fiscal state (debt, growth, primary balance, interest rate, climate, adaptation, r−g)
- $a$ ∈ {0,…,5} = six fiscal policy actions
- $r$ = multi-objective reward (debt sustainability + growth + climate resilience)
- Episode = 30-year fiscal horizon

The agent learns: *given the current fiscal position of a country, which policy instrument should
be deployed to maximise long-run welfare?*

---
## Part 1: DQN Configuration

In [ ]:
DQN_KWARGS = {
    'state_dim':             7,
    'action_dim':            6,
    'learning_rate':         1e-4,   # Adam LR — standard DQN value (Mnih et al., 2015)
    'gamma':                 0.99,   # discount — ~100 step horizon, covers 30-year episodes
    'epsilon_start':         1.0,    # start fully random
    'epsilon_end':           0.05,   # end mostly greedy
    'epsilon_decay_steps':   30_000, # decay over first 30% of training
    'buffer_size':           50_000, # ~1,667 full episodes
    'batch_size':            64,
    'target_update_freq':    500,    # hard-copy target every 500 gradient steps
    'seed':                  SEED,
}

# Print architecture info
_tmp = DQNAgent(**DQN_KWARGS)
print('DQN Architecture:')
print(_tmp.q_network)
print(f'\nTotal trainable parameters: {_tmp.count_parameters():,}')
print(f'\nHyperparameters:')
for k, v in DQN_KWARGS.items():
    print(f'  {k:<25}: {v}')
del _tmp

---
## Part 2: Train DQN Across All 9 Profiles

In [ ]:
dqn_agents = {}
dqn_logs   = {}

for profile in PROFILES:
    print(f"\n{'='*65}")
    print(f"  Training DQN: {profile}")
    print(f"{'='*65}")

    env      = make_env(profile, seed=SEED)
    eval_env = make_env(profile, seed=SEED + 1000)

    try:
        agent, log = train_dqn(
            env=env,
            total_timesteps=TOTAL_TIMESTEPS,
            agent_kwargs=DQN_KWARGS,
            eval_env=eval_env,
            eval_freq=10_000,
            eval_episodes=20,
            learning_starts=1_000,
            train_freq=4,
            verbose=True,
        )

        save_dir  = f'{MODELS_DIR}/dqn_{profile}'
        save_path = f'{save_dir}/model.pt'
        os.makedirs(save_dir, exist_ok=True)
        agent.save(save_path)

        dqn_agents[profile] = agent
        dqn_logs[profile]   = log

        recent = log['episode_rewards'][-100:] if len(log['episode_rewards']) >= 100 else log['episode_rewards']
        last_eval = log['eval_mean_rewards'][-1] if log['eval_mean_rewards'] else float('nan')
        print(f"  ✓ Done | Train(last 100ep): {np.mean(recent):>8.1f} | Final eval: {last_eval:>8.1f} | Saved: {save_path}")

    except Exception as e:
        print(f"  ✗ ERROR on {profile}: {e}")
        import traceback; traceback.print_exc()
        if profile in dqn_agents:  # save partial model if it exists
            dqn_agents[profile].save(f'{MODELS_DIR}/dqn_{profile}/model_partial.pt')

    finally:
        env.close()
        eval_env.close()

print(f"\nDQN training complete: {len(dqn_agents)}/{len(PROFILES)} profiles successful")

In [ ]:
# DQN performance summary
print('DQN TRAINING SUMMARY')
print(f'{"Profile":<30} {"Episodes":>10} {"Train(100ep)":>14} {"Final Eval":>12} {"Converged?":>12}')
print('-' * 80)
for profile in PROFILES:
    if profile not in dqn_logs:
        print(f'{profile:<30}  FAILED')
        continue
    log = dqn_logs[profile]
    eps = log['episode_rewards']
    recent = np.mean(eps[-100:]) if len(eps) >= 100 else np.mean(eps)
    early  = np.mean(eps[:100])  if len(eps) >= 100 else float('nan')
    final_eval = log['eval_mean_rewards'][-1] if log['eval_mean_rewards'] else float('nan')
    converged = 'Yes' if (not np.isnan(early) and recent > early + 20) else 'Partial'
    print(f'{profile:<30} {len(eps):>10,} {recent:>14.1f} {final_eval:>12.1f} {converged:>12}')

---
## PPO: Algorithm Explanation

### Policy Gradient Methods

Unlike DQN (which learns a value function and acts greedily), PPO directly optimises the policy
$\pi_\theta(a|s)$ by gradient ascent on the expected return:
$$J(\theta) = \mathbb{E}_{\pi_\theta}[\sum_t \gamma^t r_t]$$

The key challenge: policy gradient updates can be too large, causing the policy to collapse or
diverge. PPO addresses this with a **clipped surrogate objective**:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t,\ \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) \hat{A}_t \right) \right]$$

where $r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ is the probability
ratio (new policy / old policy) and $\hat{A}_t$ is the **generalised advantage estimate** (GAE).

The clip prevents the policy from changing too much in any single update — large policy changes
break the assumption that the collected data was generated by a policy similar to the current one.

### PPO vs DQN for This Problem

| | DQN | PPO |
|---|---|---|
| Type | Value-based | Policy-gradient |
| Policy | Deterministic (argmax Q) | Stochastic π(a\|s) |
| Sample efficiency | Higher (replay buffer) | Lower (on-policy) |
| Stability | Can oscillate | More stable updates |
| Hyperparameter sensitivity | Moderate | Lower |

Using both algorithms provides **cross-algorithmic consistency**: findings that appear in both
DQN and PPO are more robust than a single-algorithm result.

Reference: Schulman, J. et al. (2017). "Proximal Policy Optimization Algorithms." arXiv:1707.06347.

---
## Part 3: Train PPO Across All 9 Profiles

In [ ]:
# PPO metrics callback — SB3's Monitor wrapper provides episode stats
class PPOMetricsCallback(BaseCallback):
    """Collects episode rewards and lengths during PPO training.
    
    SB3's Monitor wrapper adds an 'episode' key to the info dict when an
    episode ends, containing 'r' (reward) and 'l' (length).
    """
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self) -> bool:
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
                self.episode_lengths.append(info['episode']['l'])
        return True


PPO_CONFIG = {
    'policy':        'MlpPolicy',
    'learning_rate': 3e-4,      # standard SB3 default; higher than DQN suits on-policy updates
    'n_steps':       2048,      # steps collected per rollout before update
    'batch_size':    64,
    'n_epochs':      10,        # gradient passes per rollout
    'gamma':         0.99,
    'gae_lambda':    0.95,      # GAE lambda — bias/variance trade-off in advantage estimation
    'clip_range':    0.2,       # PPO clip parameter ε (Schulman et al., 2017)
    'ent_coef':      0.01,      # entropy bonus — encourages exploration by penalising
                                # deterministic policies
    'vf_coef':       0.5,       # value function loss weight in the combined objective
    'max_grad_norm': 0.5,
    'verbose':       0,
    'seed':          SEED,
}

print('PPO Hyperparameters:')
for k, v in PPO_CONFIG.items():
    print(f'  {k:<18}: {v}')

In [ ]:
ppo_agents = {}
ppo_logs   = {}

for profile in PROFILES:
    print(f"\n{'='*65}")
    print(f"  Training PPO: {profile}")
    print(f"{'='*65}")

    save_dir = f'{MODELS_DIR}/ppo_{profile}'
    log_dir  = f'{RESULTS_DIR}/ppo_{profile}'
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    env      = Monitor(make_env(profile, seed=SEED))
    eval_env = Monitor(make_env(profile, seed=SEED + 1000))

    try:
        metrics_cb = PPOMetricsCallback()
        eval_cb    = EvalCallback(
            eval_env,
            best_model_save_path=save_dir,
            log_path=log_dir,
            eval_freq=10_000,
            n_eval_episodes=20,
            deterministic=True,
            verbose=0,
        )

        model = PPO(env=env, **PPO_CONFIG)
        model.learn(
            total_timesteps=TOTAL_TIMESTEPS,
            callback=[metrics_cb, eval_cb],
            progress_bar=False,
        )

        model.save(f'{save_dir}/final_model')
        ppo_agents[profile] = model
        ppo_logs[profile]   = {
            'episode_rewards': metrics_cb.episode_rewards,
            'episode_lengths': metrics_cb.episode_lengths,
        }

        recent = metrics_cb.episode_rewards[-100:] if len(metrics_cb.episode_rewards) >= 100 else metrics_cb.episode_rewards
        print(f"  ✓ Done | Last 100ep mean: {np.mean(recent):.1f} | Saved: {save_dir}/final_model")

    except Exception as e:
        print(f"  ✗ ERROR on {profile}: {e}")
        import traceback; traceback.print_exc()

    finally:
        env.close()
        eval_env.close()

print(f"\nPPO training complete: {len(ppo_agents)}/{len(PROFILES)} profiles successful")

In [ ]:
# PPO performance summary
print('PPO TRAINING SUMMARY')
print(f'{"Profile":<30} {"Episodes":>10} {"Train(100ep)":>14} {"Converged?":>12}')
print('-' * 70)
for profile in PROFILES:
    if profile not in ppo_logs:
        print(f'{profile:<30}  FAILED')
        continue
    eps    = ppo_logs[profile]['episode_rewards']
    recent = np.mean(eps[-100:]) if len(eps) >= 100 else np.mean(eps)
    early  = np.mean(eps[:100])  if len(eps) >= 100 else float('nan')
    converged = 'Yes' if (not np.isnan(early) and recent > early + 20) else 'Partial'
    print(f'{profile:<30} {len(eps):>10,} {recent:>14.1f} {converged:>12}')

---
## Part 4: Training Convergence Visualisation

### Figure T1: DQN Training Curves

In [ ]:
def rolling_mean(arr, window=50):
    """Compute a rolling mean over a 1-D array."""
    if len(arr) < window:
        return np.array(arr)
    return np.convolve(arr, np.ones(window)/window, mode='valid')

profile_order = [
    'Advanced_Low',         'Advanced_Medium',         'Advanced_High',
    'Emerging_Market_Low',  'Emerging_Market_Medium',  'Emerging_Market_High',
    'Developing_Low',       'Developing_Medium',       'Developing_High',
]
econ_colours = {
    'Advanced':        '#1f77b4',
    'Emerging_Market': '#ff7f0e',
    'Developing':      '#2ca02c',
}

def colour_for(profile):
    if profile.startswith('Advanced'):        return econ_colours['Advanced']
    if profile.startswith('Emerging_Market'): return econ_colours['Emerging_Market']
    return econ_colours['Developing']

# ── Figure T1: DQN training curves ───────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for idx, profile in enumerate(profile_order):
    ax = axes.flatten()[idx]
    colour = colour_for(profile)

    if profile not in dqn_logs or not dqn_logs[profile]['episode_rewards']:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(profile.replace('_', ' '), fontsize=9, fontweight='bold')
        continue

    rewards  = dqn_logs[profile]['episode_rewards']
    smoothed = rolling_mean(rewards, window=50)
    x_smooth = np.arange(len(smoothed)) + 49

    ax.plot(rewards, alpha=0.2, color=colour, linewidth=0.5)
    ax.plot(x_smooth, smoothed, color=colour, linewidth=2, label='50-ep rolling mean')

    # Convergence level (mean of last 100 episodes)
    conv_level = np.mean(rewards[-100:]) if len(rewards) >= 100 else np.mean(rewards)
    ax.axhline(conv_level, color='black', linestyle='--', linewidth=1,
               label=f'Final: {conv_level:.0f}')

    # Mark eval checkpoints
    if dqn_logs[profile]['eval_steps']:
        # Convert env steps to approx episode indices (rough mapping)
        eval_rewards = dqn_logs[profile]['eval_mean_rewards']
        ax.set_title(profile.replace('_',' '), fontsize=9, fontweight='bold')

    ax.set_xlabel('Episode', fontsize=8)
    ax.set_ylabel('Cumul. Reward', fontsize=8)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.2)
    ax.set_title(profile.replace('_',' '), fontsize=9, fontweight='bold')

# Economy-type colour legend
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color=econ_colours['Advanced'],        label='Advanced'),
    mpatches.Patch(color=econ_colours['Emerging_Market'], label='Emerging Market'),
    mpatches.Patch(color=econ_colours['Developing'],      label='Developing'),
]
fig.legend(handles=legend_patches, fontsize=10, ncol=3,
           loc='lower center', bbox_to_anchor=(0.5, -0.01))
fig.suptitle(f'DQN Training Curves — {TOTAL_TIMESTEPS:,} Timesteps per Profile',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_T1_dqn_training_curves.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_T1_dqn_training_curves')

### Figure T2: PPO Training Curves

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for idx, profile in enumerate(profile_order):
    ax = axes.flatten()[idx]
    colour = colour_for(profile)

    if profile not in ppo_logs or not ppo_logs[profile]['episode_rewards']:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(profile.replace('_',' '), fontsize=9, fontweight='bold')
        continue

    rewards  = ppo_logs[profile]['episode_rewards']
    smoothed = rolling_mean(rewards, window=50)
    x_smooth = np.arange(len(smoothed)) + 49

    ax.plot(rewards, alpha=0.2, color=colour, linewidth=0.5)
    ax.plot(x_smooth, smoothed, color=colour, linewidth=2, label='50-ep rolling mean')

    conv_level = np.mean(rewards[-100:]) if len(rewards) >= 100 else np.mean(rewards)
    ax.axhline(conv_level, color='black', linestyle='--', linewidth=1,
               label=f'Final: {conv_level:.0f}')

    ax.set_xlabel('Episode', fontsize=8)
    ax.set_ylabel('Cumul. Reward', fontsize=8)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.2)
    ax.set_title(profile.replace('_',' '), fontsize=9, fontweight='bold')

fig.legend(handles=legend_patches, fontsize=10, ncol=3,
           loc='lower center', bbox_to_anchor=(0.5, -0.01))
fig.suptitle(f'PPO Training Curves — {TOTAL_TIMESTEPS:,} Timesteps per Profile',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_T2_ppo_training_curves.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_T2_ppo_training_curves')

### Figure T3: DQN vs PPO Learning Comparison (3 Representative Profiles)

In [ ]:
representative = ['Advanced_Low', 'Emerging_Market_Medium', 'Developing_High']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, profile in zip(axes, representative):
    has_dqn = profile in dqn_logs and dqn_logs[profile]['episode_rewards']
    has_ppo = profile in ppo_logs and ppo_logs[profile]['episode_rewards']

    if has_dqn:
        dqn_r    = dqn_logs[profile]['episode_rewards']
        dqn_sm   = rolling_mean(dqn_r, window=50)
        ax.plot(dqn_sm, color='#1f77b4', linewidth=2.5,
                label=f'DQN (final: {np.mean(dqn_r[-100:]):.0f})')
        ax.fill_between(range(len(dqn_sm)), dqn_sm, alpha=0.15, color='#1f77b4')

    if has_ppo:
        ppo_r    = ppo_logs[profile]['episode_rewards']
        ppo_sm   = rolling_mean(ppo_r, window=50)
        ax.plot(ppo_sm, color='#d62728', linewidth=2.5,
                label=f'PPO (final: {np.mean(ppo_r[-100:]):.0f})')
        ax.fill_between(range(len(ppo_sm)), ppo_sm, alpha=0.15, color='#d62728')

    ax.set_title(profile.replace('_', ' '), fontsize=11, fontweight='bold')
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Reward (50-ep rolling mean)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

fig.suptitle('DQN vs PPO Learning Curves — 3 Representative Profiles',
             fontsize=13, fontweight='bold')
plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_T3_dqn_vs_ppo_learning.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_T3_dqn_vs_ppo_learning')

---
## Part 5: DQN Evaluation Curves (across training)

In [ ]:
# Plot DQN evaluation reward vs environment steps (shows true policy performance)
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for idx, profile in enumerate(profile_order):
    ax = axes.flatten()[idx]
    colour = colour_for(profile)

    if profile not in dqn_logs or not dqn_logs[profile]['eval_steps']:
        ax.text(0.5, 0.5, 'No eval data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(profile.replace('_',' '), fontsize=9, fontweight='bold')
        continue

    steps   = dqn_logs[profile]['eval_steps']
    rewards = dqn_logs[profile]['eval_mean_rewards']

    ax.plot(steps, rewards, 'o-', color=colour, linewidth=2, markersize=5)
    ax.fill_between(steps, rewards, min(rewards), alpha=0.15, color=colour)
    ax.axhline(rewards[-1], color='black', linestyle='--', linewidth=1,
               label=f'Final: {rewards[-1]:.0f}')

    ax.set_xlabel('Environment Steps', fontsize=8)
    ax.set_ylabel('Mean Eval Reward', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)
    ax.set_title(profile.replace('_',' '), fontsize=9, fontweight='bold')

fig.suptitle('DQN Evaluation Performance — Greedy Policy at Each Checkpoint',
             fontsize=13, fontweight='bold')
fig.legend(handles=legend_patches, fontsize=10, ncol=3,
           loc='lower center', bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.04, 1, 1])
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_T4_dqn_eval_curves.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_T4_dqn_eval_curves')

---
## Part 6: Save Training Logs

In [ ]:
training_summary = {
    'dqn': {
        'config':           DQN_KWARGS,
        'total_timesteps':  TOTAL_TIMESTEPS,
        'n_parameters':     18_310,  # from count_parameters() above
        'logs': {
            profile: {
                'episode_rewards':   log['episode_rewards'],
                'episode_lengths':   log['episode_lengths'],
                'eval_mean_rewards': log['eval_mean_rewards'],
                'eval_steps':        log['eval_steps'],
            }
            for profile, log in dqn_logs.items()
        },
    },
    'ppo': {
        'config':          PPO_CONFIG,
        'total_timesteps': TOTAL_TIMESTEPS,
        'logs': {
            profile: {
                'episode_rewards': log['episode_rewards'],
                'episode_lengths': log['episode_lengths'],
            }
            for profile, log in ppo_logs.items()
        },
    },
}

log_path = f'{RESULTS_DIR}/training_logs.pkl'
with open(log_path, 'wb') as f:
    pickle.dump(training_summary, f)

print(f'Training logs saved: {log_path} ({os.path.getsize(log_path):,} bytes)')

In [ ]:
# Final summary
print('=' * 70)
print('=== RL TRAINING COMPLETE ===')
print('=' * 70)
print()
print('DQN (PyTorch from scratch):')
print(f'  Architecture    : 7 → 128 → 128 → 6 (feedforward, ReLU, Huber loss)')
print(f'  Parameters      : 18,310')
print(f'  Timesteps/profile: {TOTAL_TIMESTEPS:,}')
print()
print(f'  {"Profile":<30} {"Final Train Reward":>20} {"Final Eval Reward":>18}')
print('  ' + '-' * 72)
for profile in PROFILES:
    if profile not in dqn_logs:
        print(f'  {profile:<30} FAILED')
        continue
    log  = dqn_logs[profile]
    eps  = log['episode_rewards']
    tr   = np.mean(eps[-100:]) if len(eps) >= 100 else np.mean(eps)
    ev   = log['eval_mean_rewards'][-1] if log['eval_mean_rewards'] else float('nan')
    print(f'  {profile:<30} {tr:>20.1f} {ev:>18.1f}')

print()
print('PPO (Stable-Baselines3):')
print(f'  Policy          : MlpPolicy (SB3 default: [64, 64])')
print(f'  Timesteps/profile: {TOTAL_TIMESTEPS:,}')
print()
print(f'  {"Profile":<30} {"Final Train Reward":>20}')
print('  ' + '-' * 54)
for profile in PROFILES:
    if profile not in ppo_logs:
        print(f'  {profile:<30} FAILED')
        continue
    eps = ppo_logs[profile]['episode_rewards']
    tr  = np.mean(eps[-100:]) if len(eps) >= 100 else np.mean(eps)
    print(f'  {profile:<30} {tr:>20.1f}')

print()
print('Saved locations:')
print(f'  DQN models   : outputs/models/dqn_{{profile}}/model.pt')
print(f'  PPO models   : outputs/models/ppo_{{profile}}/final_model.zip')
print(f'  Training logs: outputs/results/training_logs.pkl')
print(f'  Figures      : outputs/figures/fig_T1_dqn_training_curves.*')
print(f'                 outputs/figures/fig_T2_ppo_training_curves.*')
print(f'                 outputs/figures/fig_T3_dqn_vs_ppo_learning.*')
print(f'                 outputs/figures/fig_T4_dqn_eval_curves.*')
print('=' * 70)

if any(np.mean(dqn_logs[p]['episode_rewards'][-100:]) < np.mean(dqn_logs[p]['episode_rewards'][:100]) + 30
       for p in dqn_logs if len(dqn_logs[p]['episode_rewards']) >= 100):
    print()
    print('NOTE: Some profiles show partial convergence at 100,000 timesteps.')
    print('For stronger convergence, increase TOTAL_TIMESTEPS to 500,000.')
    print('This is documented as expected at proof-of-concept scale.')